In [0]:
import os
import sys

sys.path.append(os.getcwd())

In [0]:
import sys
if 'custom_utils' in sys.modules:
    del sys.modules['custom_utils']

from custom_utils import transformations as transformations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:

class transformations:
    def dedup(self,df:DataFrame,dedup_cols:List,cdc:str):
        df=df.withColumn('dedupkey',concat(*dedup_cols))
        df=df.withColumn("dedupCounts",row_number()\
            .over(Window.partitionBy("dedupkey").orderBy(desc(cdc))))
        df=df.filter(col('dedupCounts')==1)
        df=df.drop("dedupkey","dedupCounts")
        return df
    
    def process_timestamp(self,df):
        df=df.withColumn("process_timestamp",current_timestamp())
        return df
    def upsert(self,df,key_cols,table,cdc):
        merge_condition=" AND ".join([f"src.{i} = trg.{i}" for i in key_cols])
        dlt_obj=DeltaTable.forName(spark,f"pysparkdbt.silver.{table}")
        dlt_obj.alias("trg").merge(df.alias("src"),merge_condition)\
                            .whenMatchedUpdateAll(condition = f"src.{cdc} >= trg.{cdc}")\
                            .whenNotMatchedInsertAll()\
                            .execute()
        return 1

### customers


In [0]:
df_cust=spark.read.table("pysparkdbt.bronze.customers")

In [0]:
df_cust = df_cust.withColumn("domain",split(col("email"),"@")[1])
#df_cust.display()

In [0]:
df_cust=df_cust.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))
#df_cust.display()

In [0]:
df_cust=df_cust.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))
df_cust=df_cust.drop("first_name","last_name")
#df_cust.display()

In [0]:
cust_obj=transformations()

cust_df_trns=cust_obj.dedup(df_cust,['customer_id'],'last_updated_timestamp')
#cust_df_trns.display()


In [0]:
df_cust=cust_obj.process_timestamp(cust_df_trns)
#df_cust.display()

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.customers"):

    df_cust.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.customers")

else:

    cust_obj.upsert(df_cust,['customer_id'],'customers','last_updated_timestamp')


### Drivers


In [0]:
df_driver=spark.read.table("pysparkdbt.bronze.drivers")
display(df_driver)

driver_id,first_name,last_name,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp
1,Latasha,Lopez,262-924-2955x590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z
2,Alan,Wiley,0967969634,2,3.98,West Susan,2025-09-14T00:44:57.000Z
3,James,Taylor,424-614-1847,3,3.66,Mcintoshton,2025-08-26T22:28:17.000Z
4,Theresa,Benson,617-017-0101x91777,4,3.86,North Courtneychester,2025-09-01T11:40:55.000Z
5,Karen,Jensen,611-060-5683,5,4.87,Brownburgh,2025-09-04T16:35:04.000Z
6,Debra,Smith,556.480.9096x439,6,4.26,Port Williamland,2025-08-31T14:31:37.000Z
7,Justin,Peters,+1-798-568-6952x9778,7,3.67,West Erinborough,2025-09-17T05:57:45.000Z
8,Todd,Young,706.321.8390x08097,8,4.9,Lake Stephen,2025-08-26T06:45:51.000Z
9,Mary,Young,(172)791-0504x5499,9,4.5,West Lindsey,2025-08-27T13:04:32.000Z
10,Jacob,Mack,(509)613-4480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z


In [0]:
df_driver=df_driver.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))
#df_driver.display()

In [0]:
df_drdiver=df_driver.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))
df_driver=df_driver.drop("first_name","last_name")
#df_cust.display()

In [0]:
driver_obj=transformations()

In [0]:
df_driver = driver_obj.dedup(df_driver,['driver_id'],'last_updated_timestamp')
#df_driver.display()


In [0]:
df_driver=driver_obj.process_timestamp(df_driver)


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):

    df_driver.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.drivers")

else:

    driver_obj.upsert(df_driver,['driver_id'],'drivers','last_updated_timestamp')

In [0]:
df_loc=spark.read.table("pysparkdbt.bronze.locations")
#display(df_loc)

In [0]:
loc_obj = transformations()

In [0]:
df_loc = loc_obj.dedup(df_loc,['location_id'],'last_updated_timestamp')
df_loc = loc_obj.process_timestamp(df_loc)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.locations"):

    df_loc.write.format("delta")\
            .mode("append")\
            .saveAsTable("pysparkdbt.silver.locations")

else:

    loc_obj.upsert(df_loc,['location_id'],'locations','last_updated_timestamp')

### payments

In [0]:
df_pay = spark.read.table("pysparkdbt.bronze.payments")
#display(df_pay)


In [0]:
df_pay = df_pay.withColumn("online_payment_status",
            when( ((col('payment_method') == "Card") & (col('payment_status')=='Success')),"online_success")
            .when( ((col('payment_method') == "Card") & (col('payment_status')=='Failed')),"online_failed")
            .when( ((col('payment_method') == "Card") & (col('payment_status')=='Pending')),"online_pending")
            .otherwise("offline")
)

df_pay.display()

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,online_payment_status
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z,offline
2,676,131,Cash,Success,52.07,2025-08-14T13:00:12.000Z,2025-09-08T18:21:05.000Z,offline
3,919,132,Card,Failed,55.5,2025-07-27T13:00:12.000Z,2025-08-21T20:24:08.000Z,online_failed
4,247,34,Wallet,Pending,28.78,2025-07-27T13:00:12.000Z,2025-08-27T15:03:09.000Z,offline
5,386,62,Card,Failed,55.02,2025-08-01T13:00:12.000Z,2025-09-08T23:15:06.000Z,online_failed
6,834,78,Wallet,Success,27.43,2025-08-25T13:00:12.000Z,2025-09-20T09:03:05.000Z,offline
7,348,144,Wallet,Failed,74.94,2025-08-26T13:00:12.000Z,2025-08-30T19:11:26.000Z,offline
8,558,82,Card,Failed,19.58,2025-09-09T13:00:12.000Z,2025-09-06T05:53:26.000Z,online_failed
9,260,151,Card,Pending,24.4,2025-07-23T13:00:12.000Z,2025-09-02T17:43:14.000Z,online_pending
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z,offline


In [0]:
payment_obj = transformations()
df_pay = payment_obj.dedup(df_pay,['payment_id'],'last_updated_timestamp')
df_pay = payment_obj.process_timestamp(df_pay)



In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payments"):

    df_pay.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.payments")
    
else:

    payment_obj.upsert(df_pay,['payment_id'],'payments','last_updated_timestamp')


### vehicles

In [0]:
df_veh=spark.read.table("pysparkdbt.bronze.vehicles")
df_veh.display()

vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp
1,NXT-8646,Message,"Francis, Smith and Lee",2023,Hatchback,2025-09-02T06:05:20.000Z
2,03S R43,Region,Lawson Group,2017,Sedan,2025-08-31T05:55:09.000Z
3,SKO H06,Prepare,"Moreno, Ruiz and Barker",2023,Luxury,2025-08-30T05:07:29.000Z
4,5R235,Pattern,"Welch, Martinez and Hendricks",2019,Van,2025-09-09T05:52:10.000Z
5,925A,Process,Rivera-Anderson,2014,Hatchback,2025-08-27T11:42:16.000Z
6,6-1130V,On,Brown Ltd,2014,SUV,2025-09-06T06:37:52.000Z
7,L25-TWN,Plan,"Gonzalez, Rios and Rios",2020,Sedan,2025-09-15T13:14:48.000Z
8,GWY 958,Month,"Smith, Mckenzie and Bullock",2017,Sedan,2025-09-11T16:20:12.000Z
9,4FG 919,Speak,"Rice, Barnes and Hernandez",2019,SUV,2025-09-11T02:43:13.000Z
10,6FD 648,Religious,Schwartz and Sons,2017,SUV,2025-09-11T19:10:01.000Z


In [0]:
df_veh=df_veh.withColumn("make",upper(col("make")))
df_veh.display()

vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp
1,NXT-8646,Message,"FRANCIS, SMITH AND LEE",2023,Hatchback,2025-09-02T06:05:20.000Z
2,03S R43,Region,LAWSON GROUP,2017,Sedan,2025-08-31T05:55:09.000Z
3,SKO H06,Prepare,"MORENO, RUIZ AND BARKER",2023,Luxury,2025-08-30T05:07:29.000Z
4,5R235,Pattern,"WELCH, MARTINEZ AND HENDRICKS",2019,Van,2025-09-09T05:52:10.000Z
5,925A,Process,RIVERA-ANDERSON,2014,Hatchback,2025-08-27T11:42:16.000Z
6,6-1130V,On,BROWN LTD,2014,SUV,2025-09-06T06:37:52.000Z
7,L25-TWN,Plan,"GONZALEZ, RIOS AND RIOS",2020,Sedan,2025-09-15T13:14:48.000Z
8,GWY 958,Month,"SMITH, MCKENZIE AND BULLOCK",2017,Sedan,2025-09-11T16:20:12.000Z
9,4FG 919,Speak,"RICE, BARNES AND HERNANDEZ",2019,SUV,2025-09-11T02:43:13.000Z
10,6FD 648,Religious,SCHWARTZ AND SONS,2017,SUV,2025-09-11T19:10:01.000Z


In [0]:
vehicle_obj=transformations()
df_veh=vehicle_obj.dedup(df_veh,['vehicle_id'],'last_updated_timestamp')
df_veh=vehicle_obj.process_timestamp(df_veh)
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):

    df_veh.write.format("delta").mode("append").saveAsTable("pysparkdbt.silver.vehicles")
else:
     vehicle_obj.upsert(df_veh,['vehicle_id'],'vehicles','last_updated_timestamp')